# کوردی سۆرانی — Kurdish subtitles on a free cloud GPU

**This notebook runs the whole app on Google's free GPU, so your laptop only needs a browser.**
Nothing is installed on your computer and nothing on it gets hot.

### How to use it — three clicks

1. **Runtime → Change runtime type → T4 GPU → Save**
   *(If you skip this, everything still works, but on a slow CPU.)*
2. Press **▶ Run all** (or run the cells top to bottom).
3. Wait until a box prints **“OPEN THIS LINK”** — click that link.
   The normal Kurdish Sub page opens in a new tab, running on the GPU.

**One permission window will appear** (cell 3): it asks whether the notebook may
save your results to Google Drive. **Allow** is recommended — but **Cancel**
changes nothing about the app; you simply download the files instead.

Keep this Colab tab open while you work. When you are done, close it: the free
machine is deleted automatically.

> Free Colab gives about **12 hours** per session and disconnects after ~90
> minutes of doing nothing. For a normal film you need 10–40 minutes.

## 1 · Check the GPU

You want to see a **Tesla T4**. If it says “command not found”, the GPU is not
switched on: Runtime → Change runtime type → **T4 GPU** → Save, then run this
cell again.

In [1]:
!nvidia-smi

Mon Nov  2 02:36:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2 · Install Kurdish Sub

Two to four minutes the first time. Everything is downloaded into this temporary
machine; the speech model is downloaded again next time (it is ~1.5 GB, once).

In [ ]:
import os, pathlib, subprocess, sys, shutil

REPO = "https://github.com/sahandking86-lgtm/kurdish-sub.git"
# Put the arena/cloud branch first so we download the version containing cloud files
BRANCHES = ["arena/190df544-kurdish-sub", "main"]
DEST = pathlib.Path("/content/kurdish-sub")

# Switch to a safe root directory before doing any file/directory deletions
os.chdir("/content")

# Force a redownload if we do not have the cloud folder
if DEST.exists() and not (DEST / "cloud" / "start_cloud.py").exists():
    print("Cleaning up incomplete or wrong branch repository installation...")
    shutil.rmtree(DEST, ignore_errors=True)

if not DEST.exists():
    for branch in BRANCHES:
        print(f"downloading [{branch}] …")
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", branch, REPO, str(DEST)],
                           capture_output=True, text=True)
        if r.returncode == 0:
            break
        print(f"  Failed to clone branch {branch}:")
        print(r.stderr)
    else:
        raise SystemExit("Could not download the app — check the internet connection or the repository URL.")

os.chdir(DEST)
print("Contents of downloaded repository:", os.listdir('.'))

print("installing the Python packages (a few minutes) …")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("packages:", "ok" if r.returncode == 0 else "FAILED")

print("Downgrading PyAV (av) to a version older than 19.0.0 …")
r_av = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av<19.0.0"])
print("PyAV downgrade:", "ok" if r_av.returncode == 0 else "FAILED")

print("Upgrading soundfile package to avoid metadata_errors …")
r_sf = subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-q", "soundfile"])
print("soundfile upgrade:", "ok" if r_sf.returncode == 0 else "FAILED")

# make sure the GPU can really run the speech model
if pathlib.Path("cloud/doctor.py").exists():
    !python cloud/doctor.py --fix
else:
    print("Warning: cloud/doctor.py not found!")

downloading [arena/190df544-kurdish-sub] …
Contents of downloaded repository: ['start-linux.sh', 'README.md', 'app', 'tests', 'requirements.txt', '.gitignore', 'Translate.zip', 'SETUP.md', 'docs', 'requirements-dev.txt', 'requirements-offline.txt', 'start-windows.bat', '.git', 'cloud']
installing the Python packages (a few minutes) …
packages: ok
Downgrading PyAV (av) to a version older than 19.0.0 …
PyAV downgrade: ok
Upgrading soundfile package to avoid metadata_errors …
soundfile upgrade: ok

  Kurdish Sub — machine check
  ────────────────────────────────────────
  ✅ Python 3.13.16
  ✅ fastapi installed
  ✅ faster_whisper installed
  ✅ imageio_ffmpeg installed
  ✅ NVIDIA GPU: Tesla T4, 15360 MiB
  ✅ ffmpeg: /usr/bin/ffmpeg
  ✅ video encoder: h264_nvenc (hardware — fast)
  ✅ free disk space: 112 GB
  ✅ huggingface.co (speech models)
  ❌ translate.googleapis.com (free translation)


## 3 · Connect Google Drive — recommended (20 seconds)

A finished film is big. Connecting Drive means a copy of every video and `.srt`
is saved there automatically, so nothing is lost when the free machine switches
off — and you can watch it on your phone.

A Google window will ask for permission: choose your account → **Allow**.
If you would rather not, just run the cell and close the pop-up; everything still
works and you download the result from the app.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive not connected — that is fine:", exc)

## 4 · Start it and get your link

This prints a **private link**. Open it in a new tab (or on your phone) and use
the app exactly like the desktop version — drop a video, pick the accuracy,
wait, download the result.

*Included:* `--drive` copies every finished video and `.srt` into
`My Drive → KurdishSub` **if** you have connected Drive in this session.
If you have not, results are still downloadable from the app, and from the
folder icon on the left of this notebook (`kurdish-sub-cloud/output/…`).

In [ ]:
import os, pathlib

# Ensure we are in the correct directory where the repository is cloned
if pathlib.Path('/content/kurdish-sub').exists():
    os.chdir('/content/kurdish-sub')

if not pathlib.Path('cloud/start_cloud.py').exists():
    raise FileNotFoundError("Could not find 'cloud/start_cloud.py'. Please make sure Cell 2 completed successfully.")

%run cloud/start_cloud.py --drive

## 5 · Optional — better Kurdish (1 minute)

Kurdish text from **Google Gemini** is clearly better than the keyless Google
Translate engine: more natural sentences, keeps slang and profanity.

Get a free key at <https://aistudio.google.com/apikey>, then paste it in the
app's **gear icon → Translation engine → Google Gemini**. That is all — no need
to touch this notebook.

> Gemini's free tier is not available in every country. If the key is refused,
> stay on *Automatic* (Google Translate works everywhere).

## 6 · Keep the session awake (only for long films)

Colab disconnects a session after about 90 minutes without activity. A normal
film finishes long before that. If you are doing a **very long video** (a 3-hour
film, “Perfect” accuracy), run this cell and leave it running — it prints a dot
every five minutes so Google knows you are still there.

Press the **■ stop** button of this cell when you are finished. The app keeps
working normally while this cell runs.

In [ ]:
KEEP_AWAKE = False   # ← change to True only for a very long film

import datetime, time

if not KEEP_AWAKE:
    print("Skipped (this cell is only for very long videos).")
else:
    print("Keeping the session awake — press the stop button when you are done.")
    try:
        while True:
            time.sleep(300)
            print("still awake", datetime.datetime.now().strftime("%H:%M"), flush=True)
    except KeyboardInterrupt:
        print("stopped")


## When you are finished

* **Download** the finished video from the app, or from the file browser on the left.
* Close the browser tab. Google deletes the machine and the models after a while.
* Next time: open this notebook again and press **Run all** — same steps, fresh GPU.

### If something looks wrong

| What you see | What to do |
|---|---|
| `command not found: nvidia-smi` | Runtime → Change runtime type → **T4 GPU** → Save |
| “No GPU available” in Colab | Google is busy. Wait a few minutes, or use a Kaggle account (30 free GPU hours/week) |
| The link does not open | Use the interface shown inside this notebook instead (it is the same app) |
| The notebook disconnected | Run the cells again — that is normal on the free tier |
| Everything is very slow | Check the first cell says **Tesla T4**; pick the “Medium” accuracy level |